# Judy Hazem - RAG Integration

This notebook covers milestone 4: combine retrieval with a generation layer that produces citation-grounded answers from the retrieved StackLite contexts.

## 1. Setup

The notebook reuses the project modules so the script, tests, reports, and saved notebook outputs stay aligned.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'stacklite_questions.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_rag_demo import ANSWERS_OUTPUT, CONTEXTS_OUTPUT, write_answer_outputs, write_context_outputs
from src.bm25_retriever import BM25Retriever, load_stacklite_dataset
from src.evaluation import load_evaluation_questions
from src.rag_pipeline import CitationAnswerGenerator, HybridRAGPipeline
from src.semantic_retriever import DEFAULT_MINILM_MODEL, SemanticRetriever

DATASET_PATH = PROJECT_ROOT / 'data' / 'stacklite_questions.csv'
QUESTIONS_PATH = PROJECT_ROOT / 'evaluation' / 'bm25_eval_queries.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_PATH}')
print(f'MiniLM model: {DEFAULT_MINILM_MODEL}')

Project root: C:\Users\RTX\Downloads\Advanced DL
Dataset: C:\Users\RTX\Downloads\Advanced DL\data\stacklite_questions.csv
MiniLM model: sentence-transformers/all-MiniLM-L6-v2


## 2. Load Corpus And Questions

In [2]:
documents = load_stacklite_dataset(DATASET_PATH)
questions = load_evaluation_questions(QUESTIONS_PATH)

print(f'Loaded documents: {len(documents)}')
print(f'Questions for RAG demo: {len(questions)}')
for question in questions:
    print(f'- {question.query_id}: {question.query}')

Loaded documents: 1500
Questions for RAG demo: 5
- q1_micro_macro_average: Why are micro average precision recall and F1 equal in multiclass classification?
- q2_ai_vs_ml: What is the difference between artificial intelligence and machine learning?
- q3_deconvolution_layers: What are deconvolutional layers in convolutional neural networks?
- q4_keras_class_weights: How do I set class weights for imbalanced classes in Keras?
- q5_dying_relu: What is the dying ReLU problem in neural networks?


## 3. Build Hybrid Retrieval

In [3]:
bm25_retriever = BM25Retriever(k1=1.5, b=0.75).fit(documents)
semantic_retriever = SemanticRetriever(model_name=DEFAULT_MINILM_MODEL).fit(documents)

pipeline = HybridRAGPipeline(
    bm25_retriever=bm25_retriever,
    semantic_retriever=semantic_retriever,
    generator=CitationAnswerGenerator(max_contexts=3),
    candidate_k=10,
    context_k=3,
)

print('BM25 index ready')
print('MiniLM semantic index ready')
print('Hybrid RAG pipeline ready')

BM25 index ready
MiniLM semantic index ready
Hybrid RAG pipeline ready


## 4. Generate Cited Answers

In [4]:
responses = [pipeline.answer(question.query) for question in questions[:5]]

for response in responses:
    print('=' * 80)
    print(f'Question: {response.question}')
    print(response.answer)
    print('Citations:', ', '.join(citation['citation'] for citation in response.citations))

Question: Why are micro average precision recall and F1 equal in multiclass classification?
The retrieved StackLite evidence points to the following cited source passages as the best grounding for the answer.

[1] Micro Average vs Macro average Performance in a Multiclass classification setting: Micro Average vs Macro average Performance in a Multiclass classification setting Micro Average vs Macro average Performance in a Multiclass classification setting I am trying out...
[2] Why is the F-measure preferred for classification tasks?: Why is the F-measure preferred for classification tasks? Why is the F-measure preferred for classification tasks? Why is the F-measure usually used for (supervised) classification...
[3] What's the difference between Sklearn F1 score 'micro' and 'weighted' for a multi class classification problem?: What's the difference between Sklearn F1 score 'micro' and 'weighted' for a multi class classification problem? What's the difference between Sklearn F1 score

## 5. Save Demo Outputs

In [5]:
write_answer_outputs(responses, ANSWERS_OUTPUT)
write_context_outputs(responses, CONTEXTS_OUTPUT)

print(f'Answers CSV: {ANSWERS_OUTPUT}')
print(f'Retrieved contexts CSV: {CONTEXTS_OUTPUT}')

Answers CSV: C:\Users\RTX\Downloads\Advanced DL\results\rag_sample_answers.csv
Retrieved contexts CSV: C:\Users\RTX\Downloads\Advanced DL\results\rag_retrieved_contexts.csv


## 6. Inspect Answers Table

In [6]:
import pandas as pd

pd.read_csv(ANSWERS_OUTPUT)[['question', 'answer', 'citations']]

,question,answer,citations
0,Why are micro average precision recall and F1 ...,The retrieved StackLite evidence points to the...,[1] [2] [3]
1,What is the difference between artificial inte...,The retrieved StackLite evidence points to the...,[1] [2] [3]
2,What are deconvolutional layers in convolution...,The retrieved StackLite evidence points to the...,[1] [2] [3]
3,How do I set class weights for imbalanced clas...,The retrieved StackLite evidence points to the...,[1] [2] [3]
4,What is the dying ReLU problem in neural netwo...,The retrieved StackLite evidence points to the...,[1] [2] [3]
